# 01 — Data Cleaning and Validation

This notebook creates a validated, chronologically ordered dataset for the Dublin Bikes forecasting project.

Main corrections:

- Uses portable `pathlib` paths.
- Parses timestamps explicitly.
- Removes exact duplicates.
- Validates bike-capacity business rules.
- Preserves station identity.
- Does **not** scale features or the target.
- Writes a clean intermediate dataset for later notebooks.

In [4]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "Data").exists() or (candidate / "data").exists():
            return candidate.resolve()
    return Path.cwd().resolve()

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "Data"
OUTPUT_DIR = PROJECT_ROOT / "Outputs"
FIGURE_DIR = OUTPUT_DIR / "Figures"
MODEL_DIR = OUTPUT_DIR / "Models"

for directory in (DATA_DIR, FIGURE_DIR, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

Project root: /mnt/WorkSpace/Repos/dublin-bikes-fix


In [5]:
RAW_PATH = DATA_DIR / "dataset.csv"
CLEAN_PATH = DATA_DIR / "dataset_cleaned.csv"

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Expected raw data at {RAW_PATH}. "
        "Place the Dublin Bikes CSV there before running this notebook."
    )

df_raw = pd.read_csv(RAW_PATH)
print(f"Raw shape: {df_raw.shape}")
display(df_raw.head())

Raw shape: (158981, 11)


,STATION ID,TIME,LAST UPDATED,NAME,BIKE_STANDS,AVAILABLE_BIKE_STANDS,AVAILABLE_BIKES,STATUS,ADDRESS,LATITUDE,LONGITUDE
0,2,2021-11-01 00:00:02,2021-10-31 23:58:20,BLESSINGTON STREET,20,10,10,OPEN,Blessington Street,53.3568,-6.26814
1,3,2021-11-01 00:00:02,2021-10-31 23:56:16,BOLTON STREET,20,8,12,OPEN,Bolton Street,53.3512,-6.26986
2,4,2021-11-01 00:00:02,2021-10-31 23:54:38,GREEK STREET,20,10,10,OPEN,Greek Street,53.3469,-6.27298
3,5,2021-11-01 00:00:02,2021-10-31 23:50:09,CHARLEMONT PLACE,40,3,37,OPEN,Charlemont Street,53.3307,-6.26018
4,6,2021-11-01 00:00:02,2021-10-31 23:57:50,CHRISTCHURCH PLACE,20,9,11,OPEN,Christchurch Place,53.3434,-6.27012


## Standardize schema

In [6]:
df = df_raw.copy()
df.columns = [column.strip().upper().replace("_", " ") for column in df.columns]

rename_map = {
    "STATIONID": "STATION ID",
    "STATION ID": "STATION ID",
    "LASTUPDATED": "LAST UPDATED",
    "LAST UPDATED": "LAST UPDATED",
    "BIKESTANDS": "BIKE STANDS",
    "BIKE STANDS": "BIKE STANDS",
    "AVAILABLEBIKESTANDS": "AVAILABLE BIKE STANDS",
    "AVAILABLE BIKE STANDS": "AVAILABLE BIKE STANDS",
    "AVAILABLEBIKES": "AVAILABLE BIKES",
    "AVAILABLE BIKES": "AVAILABLE BIKES",
    "LAT": "LATITUDE",
    "LNG": "LONGITUDE",
    "LON": "LONGITUDE",
}
df = df.rename(columns={column: rename_map.get(column.replace(" ", ""), column) for column in df.columns})

required_columns = {
    "STATION ID", "TIME", "BIKE STANDS",
    "AVAILABLE BIKE STANDS", "AVAILABLE BIKES",
    "STATUS", "LATITUDE", "LONGITUDE"
}
missing = required_columns.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

print("Columns:", list(df.columns))

Columns: ['STATION ID', 'TIME', 'LAST UPDATED', 'NAME', 'BIKE STANDS', 'AVAILABLE BIKE STANDS', 'AVAILABLE BIKES', 'STATUS', 'ADDRESS', 'LATITUDE', 'LONGITUDE']


## Parse types and remove unusable records

In [7]:
for column in ["TIME", "LAST UPDATED"]:
    if column in df.columns:
        df[column] = pd.to_datetime(df[column], errors="coerce", utc=False)

numeric_columns = [
    "STATION ID", "BIKE STANDS", "AVAILABLE BIKE STANDS",
    "AVAILABLE BIKES", "LATITUDE", "LONGITUDE"
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

before = len(df)
df = df.dropna(subset=[
    "STATION ID", "TIME", "BIKE STANDS",
    "AVAILABLE BIKE STANDS", "AVAILABLE BIKES",
    "LATITUDE", "LONGITUDE"
])
print(f"Dropped {before - len(df):,} rows with invalid required values.")

duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates()
print(f"Removed {duplicate_count:,} exact duplicate rows.")

Dropped 0 rows with invalid required values.
Removed 0 exact duplicate rows.


## Apply business-rule validation

In [8]:
rule_masks = {
    "negative_capacity": df["BIKE STANDS"] < 0,
    "negative_available_bikes": df["AVAILABLE BIKES"] < 0,
    "negative_available_stands": df["AVAILABLE BIKE STANDS"] < 0,
    "bikes_above_capacity": df["AVAILABLE BIKES"] > df["BIKE STANDS"],
    "stands_above_capacity": df["AVAILABLE BIKE STANDS"] > df["BIKE STANDS"],
}

rule_summary = pd.Series(
    {name: int(mask.sum()) for name, mask in rule_masks.items()},
    name="invalid_rows"
).to_frame()
display(rule_summary)

invalid_mask = np.logical_or.reduce(list(rule_masks.values()))
invalid_rows = df.loc[invalid_mask].copy()

if not invalid_rows.empty:
    invalid_path = DATA_DIR / "invalid_rows.csv"
    invalid_rows.to_csv(invalid_path, index=False)
    print(f"Saved {len(invalid_rows):,} invalid rows to {invalid_path}")

df = df.loc[~invalid_mask].copy()

,invalid_rows
negative_capacity,0
negative_available_bikes,0
negative_available_stands,0
bikes_above_capacity,0
stands_above_capacity,0


## Check sensor consistency

In [9]:
df["CAPACITY_DIFFERENCE"] = (
    df["BIKE STANDS"]
    - (df["AVAILABLE BIKES"] + df["AVAILABLE BIKE STANDS"])
)

consistency_summary = df["CAPACITY_DIFFERENCE"].describe()
display(consistency_summary.to_frame("capacity_difference"))

# Keep the observations because maintenance/reserved stands can explain a difference.
# The difference is retained as a data-quality feature for analysis but should not be
# used as a predictor if it relies on contemporaneous availability values.

,capacity_difference
count,158981.000000
mean,0.067562
std,0.304982
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,8.000000


## Final cleaning

In [10]:
df["STATION ID"] = df["STATION ID"].astype("int64")
df["BIKE STANDS"] = df["BIKE STANDS"].astype("int64")
df["AVAILABLE BIKE STANDS"] = df["AVAILABLE BIKE STANDS"].astype("int64")
df["AVAILABLE BIKES"] = df["AVAILABLE BIKES"].astype("int64")
df["STATUS"] = df["STATUS"].astype(str).str.strip().str.upper()

df = (
    df.sort_values(["TIME", "STATION ID"])
      .reset_index(drop=True)
)

assert df["TIME"].is_monotonic_increasing
assert (df["AVAILABLE BIKES"] >= 0).all()
assert (df["AVAILABLE BIKES"] <= df["BIKE STANDS"]).all()

df.to_csv(CLEAN_PATH, index=False)

print(f"Clean dataset shape: {df.shape}")
print(f"Date range: {df['TIME'].min()} to {df['TIME'].max()}")
print(f"Stations: {df['STATION ID'].nunique()}")
print(f"Saved to: {CLEAN_PATH}")
display(df.head())

Clean dataset shape: (158981, 12)
Date range: 2021-11-01 00:00:02 to 2021-11-30 23:30:02
Stations: 111
Saved to: /mnt/WorkSpace/Repos/dublin-bikes-fix/Data/dataset_cleaned.csv


,STATION ID,TIME,LAST UPDATED,NAME,BIKE STANDS,AVAILABLE BIKE STANDS,AVAILABLE BIKES,STATUS,ADDRESS,LATITUDE,LONGITUDE,CAPACITY_DIFFERENCE
0,2,2021-11-01 00:00:02,2021-10-31 23:58:20,BLESSINGTON STREET,20,10,10,OPEN,Blessington Street,53.3568,-6.26814,0
1,3,2021-11-01 00:00:02,2021-10-31 23:56:16,BOLTON STREET,20,8,12,OPEN,Bolton Street,53.3512,-6.26986,0
2,4,2021-11-01 00:00:02,2021-10-31 23:54:38,GREEK STREET,20,10,10,OPEN,Greek Street,53.3469,-6.27298,0
3,5,2021-11-01 00:00:02,2021-10-31 23:50:09,CHARLEMONT PLACE,40,3,37,OPEN,Charlemont Street,53.3307,-6.26018,0
4,6,2021-11-01 00:00:02,2021-10-31 23:57:50,CHRISTCHURCH PLACE,20,9,11,OPEN,Christchurch Place,53.3434,-6.27012,0
